In [1]:
using LinearAlgebra
using Plots
using Statistics

gr()
Plots.GRBackend()

include("Optimisation.jl")
include("Parameterization.jl")
include("RaySolver.jl")
include("PlotRays.jl")
include("StatisticsRays.jl")


repartition (generic function with 1 method)

In [2]:
controlx = [-1.5 -1.2 -0.7 -0.3 0. 0.3 0.7 1.2 1.5]
controly = [0. 0.3 0.5 0.7 1. 0.7 0.5 0.3 0.] .+ 1.
control = vcat(controlx, controly)
Bezier(control, 0.5)
a = map( i-> Bezier(control,i/100), 1:100)

100-element Vector{Vector{Float64}}:
 [-1.4754565911172077, 1.0237255956034095]
 [-1.449891075661824, 1.0469246619406027]
 [-1.4233968601362954, 1.0696301713110064]
 [-1.3960625755422715, 1.0918741469416648]
 [-1.3679722068749995, 1.1136872668281248]
 [-1.3392052207226874, 1.135098519387719]
 [-1.3098366910111432, 1.156134908150449]
 [-1.2799374229340164, 1.176821202772296]
 [-1.2495740751089526, 1.1971797337154995]
 [-1.2188092800000003, 1.2172302280000002]
 [-1.1877017626465682, 1.2369896834899121]
 [-1.156306457739264, 1.2564722792385743]
 [-1.1246746250829358, 1.2756893194754029]
 ⋮
 [1.187701762646568, 1.236989683489912]
 [1.2188092799999999, 1.217230228]
 [1.2495740751089521, 1.1971797337154992]
 [1.2799374229340161, 1.1768212027722957]
 [1.309836691011144, 1.1561349081504493]
 [1.3392052207226879, 1.1350985193877197]
 [1.367972206875, 1.113687266828125]
 [1.3960625755422718, 1.0918741469416653]
 [1.4233968601362958, 1.0696301713110064]
 [1.449891075661824, 1.046924661940603]
 [1

On définit un type appelé ray (struct) :

In [3]:
ray = Ray([-0.5, 0.], pi/2.)

Ray([-0.5, 0.0], 1.5707963267948966)

On peut utiuliser les générateurs pour éviter d'avoir à stocker un grand nombre de rayons

In [4]:
Nx = 10
Ntheta = 5
#angles = (-0.2 + i / Ntheta * 0.4 for i in 0:Ntheta)
angles = Iterators.map( i -> -0.2 + i / Ntheta * 0.4, 0:Ntheta)
#Il faut créer les couleurs ici et les associer aux bonnes directions de rayons.
ray = Ray([-0.5, 0.], pi/2)

Ray([-0.5, 0.0], 1.5707963267948966)

Les rays sont sous forme de matrices dont les lignes correspondent à chaque position et les colonnes aux vitesses

In [5]:
rays = (Ray([-0.5 +  i / Nx, 0. ], theta) for i in 0:Nx, theta in angles )
length(rays)
collect(rays)

11×6 Matrix{Ray}:
 Ray([-0.5, 0.0], -0.2)  Ray([-0.5, 0.0], -0.12)  …  Ray([-0.5, 0.0], 0.2)
 Ray([-0.4, 0.0], -0.2)  Ray([-0.4, 0.0], -0.12)     Ray([-0.4, 0.0], 0.2)
 Ray([-0.3, 0.0], -0.2)  Ray([-0.3, 0.0], -0.12)     Ray([-0.3, 0.0], 0.2)
 Ray([-0.2, 0.0], -0.2)  Ray([-0.2, 0.0], -0.12)     Ray([-0.2, 0.0], 0.2)
 Ray([-0.1, 0.0], -0.2)  Ray([-0.1, 0.0], -0.12)     Ray([-0.1, 0.0], 0.2)
 Ray([0.0, 0.0], -0.2)   Ray([0.0, 0.0], -0.12)   …  Ray([0.0, 0.0], 0.2)
 Ray([0.1, 0.0], -0.2)   Ray([0.1, 0.0], -0.12)      Ray([0.1, 0.0], 0.2)
 Ray([0.2, 0.0], -0.2)   Ray([0.2, 0.0], -0.12)      Ray([0.2, 0.0], 0.2)
 Ray([0.3, 0.0], -0.2)   Ray([0.3, 0.0], -0.12)      Ray([0.3, 0.0], 0.2)
 Ray([0.4, 0.0], -0.2)   Ray([0.4, 0.0], -0.12)      Ray([0.4, 0.0], 0.2)
 Ray([0.5, 0.0], -0.2)   Ray([0.5, 0.0], -0.12)   …  Ray([0.5, 0.0], 0.2)

On utilise map dans ce qui suit mais la syntaxe précédente fonctionne aussi

In [6]:
refrays = Iterators.map( ray-> PropagateRay(control, ray), rays);
Xf = Iterators.map( ray -> ProjectionFocalPlane(ray), refrays);

Calcul des quantiles des angles d'incidence

In [7]:
println(collect(angles))
quantile(angles,1/6)

[-0.2, -0.12, -0.03999999999999998, 0.03999999999999998, 0.12000000000000005, 0.2]


-0.13333333333333333

In [8]:
PlotRays(control,X,Xi)

LoadError: UndefVarError: `X` not defined

In [857]:
#focaldircondexpectancy(control, X, [0.1,1.])
sum([xf,xf^2] for xf in Xf) / length(Xf)

2-element Vector{Float64}:
 2.517512527494686e-18
 0.35346557487447733

In [754]:
focaleEVar(control, X, Xi)
#focalexpectancy(control, X, [0.1,1.])

(-9.175396897728567e-17, 0.3705866952953192)

In [755]:
rmsspot(control,X,Xi)

0.3694783568259012

In [797]:
ray0 = Ray([0.,0.], [0. , 1.])
dray0 = Ray([0.1,0.], [0. , 0.])
BackwardBilliard(control, ray0, dray0)

Ray([0.10000000127009107, -4.440892098500626e-9], [-0.1999650711779328, 0.0])

In [798]:
T = xmin .+ (xmax - xmin) .* [k for k in 0:N] ./ N
dist = repartition(control, X, Xi)
plt = plot(T,dist, aspect_ratio=:equal, label = "Distribution")
display(plt)


LoadError: UndefVarError: `xmin` not defined